In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Dữ liệu/student_data/shipments_realistic.csv")
df.shape

Mounted at /content/drive


(566067, 22)

In [2]:
# ============================================================
# 1. IMPORT THƯ VIỆN
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 2. ĐỌC DỮ LIỆU GỐC
# ============================================================

file_path = "/content/drive/MyDrive/Colab Notebooks/Dữ liệu/student_data/shipments_realistic.csv"

df = pd.read_csv(file_path)

print("Đọc dữ liệu gốc thành công!")
print("Kích thước dữ liệu:", df.shape)

display(df.head())


# ============================================================
# 3. KIỂM TRA DỮ LIỆU GỐC
# ============================================================

print("========== KIỂM TRA DỮ LIỆU GỐC ==========")

print("\n1. Thông tin dữ liệu:")
df.info()

print("\n2. Giá trị thiếu:")
print(df.isnull().sum())

print("\n3. Số dòng trùng hoàn toàn:")
print(df.duplicated().sum())

print("\n4. Số đơn hàng:")
print(df["order_id"].nunique())


# ============================================================
# 4. SAO CHÉP DỮ LIỆU
# ============================================================

df_clean = df.copy()


# ============================================================
# 5. CHUẨN HÓA TÊN CỘT
# ============================================================

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
)


# ============================================================
# 6. CHUYỂN ĐỔI KIỂU DỮ LIỆU
# ============================================================

# Chuyển ngày gửi hàng
df_clean["ship_date"] = pd.to_datetime(
    df_clean["ship_date"],
    errors="coerce"
)


# Chuyển ngày giao hàng
df_clean["delivery_date"] = pd.to_datetime(
    df_clean["delivery_date"],
    errors="coerce"
)


# Chuyển order_id sang số nguyên
df_clean["order_id"] = pd.to_numeric(
    df_clean["order_id"],
    errors="coerce"
)


# Chuyển shipping_fee sang số thực
df_clean["shipping_fee"] = pd.to_numeric(
    df_clean["shipping_fee"],
    errors="coerce"
)


# Chuẩn hóa shipper_id
df_clean["shipper_id"] = (
    df_clean["shipper_id"]
    .astype("string")
    .str.strip()
)


# ============================================================
# 7. KIỂM TRA GIÁ TRỊ THIẾU
# ============================================================

print("\n========== GIÁ TRỊ THIẾU ==========")
print(df_clean.isnull().sum())


# ============================================================
# 8. XÓA CÁC DÒNG THIẾU DỮ LIỆU QUAN TRỌNG
# ============================================================

required_columns = [
    "order_id",
    "ship_date",
    "delivery_date",
    "shipping_fee",
    "shipper_id"
]

df_clean = df_clean.dropna(
    subset=required_columns
)


# ============================================================
# 9. XÓA DÒNG TRÙNG LẶP HOÀN TOÀN
# ============================================================

df_clean = df_clean.drop_duplicates()


# ============================================================
# 10. KIỂM TRA KHÓA CHÍNH
# ============================================================

# Theo sơ đồ, order_id là khóa chính của Shipment.
# Mỗi đơn hàng chỉ có một bản ghi vận chuyển.

print("\nSố dòng bị trùng order_id:")
print(df_clean["order_id"].duplicated().sum())

# Nếu có đơn hàng trùng, giữ lại bản ghi đầu tiên
df_clean = df_clean.drop_duplicates(
    subset=["order_id"],
    keep="first"
)


# ============================================================
# 11. LOẠI BỎ DỮ LIỆU KHÔNG HỢP LỆ
# ============================================================

# order_id phải lớn hơn 0
df_clean = df_clean[
    df_clean["order_id"] > 0
]


# Phí vận chuyển không được âm
df_clean = df_clean[
    df_clean["shipping_fee"] >= 0
]


# Ngày giao hàng phải lớn hơn hoặc bằng ngày gửi hàng
df_clean = df_clean[
    df_clean["delivery_date"] >= df_clean["ship_date"]
]


# ============================================================
# 12. KIỂM TRA SHIPPER_ID
# ============================================================

# Mã shipper phải có dạng SHPxxxxx
# Ví dụ: SHP00001

invalid_shipper_id = ~df_clean["shipper_id"].str.match(
    r"^SHP\d+$",
    na=False
)

print("\nSố shipper_id không đúng định dạng:")
print(invalid_shipper_id.sum())

df_clean = df_clean[
    ~invalid_shipper_id
]


# ============================================================
# 13. CHUYỂN KIỂU DỮ LIỆU
# ============================================================

df_clean["order_id"] = (
    df_clean["order_id"].astype(int)
)


# ============================================================
# 14. TẠO DATAFRAME MỚI TÊN SHIPMENT
# ============================================================

shipment_columns = [
    "order_id",
    "ship_date",
    "delivery_date",
    "shipping_fee",
    "shipper_id"
]

Shipment = df_clean[shipment_columns].copy()


# ============================================================
# 15. ĐỊNH DẠNG NGÀY
# ============================================================

Shipment["ship_date"] = (
    Shipment["ship_date"].dt.date
)

Shipment["delivery_date"] = (
    Shipment["delivery_date"].dt.date
)


# ============================================================
# 16. SẮP XẾP DỮ LIỆU
# ============================================================

Shipment = Shipment.sort_values(
    by="order_id"
).reset_index(drop=True)


# ============================================================
# 17. KIỂM TRA KẾT QUẢ
# ============================================================

print("\n========== KẾT QUẢ BẢNG SHIPMENT ==========")

print("\n1. Kích thước dữ liệu:")
print(Shipment.shape)

print("\n2. Số đơn hàng:")
print(Shipment["order_id"].nunique())

print("\n3. Giá trị thiếu:")
print(Shipment.isnull().sum())

print("\n4. Số khóa chính bị trùng:")
print(Shipment["order_id"].duplicated().sum())

print("\n5. Các cột:")
print(Shipment.columns.tolist())

print("\n6. Năm dòng đầu tiên:")
display(Shipment.head())


# ============================================================
# 18. KIỂM TRA TÍNH HỢP LỆ
# ============================================================

assert Shipment["order_id"].is_unique
assert Shipment["order_id"].notna().all()

assert Shipment["shipping_fee"].ge(0).all()

assert (
    Shipment["delivery_date"] >= Shipment["ship_date"]
).all()

assert Shipment["shipper_id"].notna().all()

print("\nDữ liệu SHIPMENT đã vượt qua các kiểm tra cơ bản!")


# ============================================================
# 19. LƯU FILE CSV
# ============================================================

shipment_output_path = "/content/Shipment_clean.csv"

Shipment.to_csv(
    shipment_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nĐã lưu file:")
print(shipment_output_path)


# ============================================================
# 20. HIỂN THỊ TOÀN BỘ BẢNG SHIPMENT
# ============================================================

display(Shipment)

Đọc dữ liệu gốc thành công!
Kích thước dữ liệu: (566067, 22)


,shipper_id,order_id,ship_date,delivery_date,shipping_fee,shipper_company,shipper_vehicle,shipper_experience_years,shipper_rating,delivery_success_rate,...,join_date,shipper_name,shipper_phone,shipper_gender,shipper_age,shipper_marital_status,shipper_education,city,region,district
0,SHP00001,1,2012-07-07,2012-07-11,1.37,Viettel Post,Truck,7,5.0,99.0,...,2026-03-17,Bùi Văn Long,991476209,Male,27,Married,Bachelor,Phan Rang-Thap Cham,Central,District #25
1,SHP00002,2,2012-07-06,2012-07-10,2.60,J&T Express,Van,2,4.9,98.4,...,2025-01-29,Trần Anh Khánh,959297982,Male,41,Married,Bachelor,Phan Thiet,Central,District #29
2,SHP00003,3,2012-07-04,2012-07-07,2.38,GHN,Motorbike,10,4.8,95.1,...,2019-11-13,Hoàng Thị Khánh,927142576,Male,30,Single,High School,Long Xuyen,West,District #34
3,SHP00004,4,2012-07-05,2012-07-11,2.49,Viettel Post,Truck,8,5.0,96.3,...,2025-12-22,Trần Đức Vy,971617475,Female,42,Married,College,Kon Tum,Central,District #27
4,SHP00005,6,2012-07-09,2012-07-16,25.79,BEST Express,Truck,10,4.6,95.7,...,2019-12-19,Trần Minh Cường,979196342,Male,31,Married,College,Da Nang,Central,District #23


========== KIỂM TRA DỮ LIỆU GỐC ==========

1. Thông tin dữ liệu:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566067 entries, 0 to 566066
Data columns (total 22 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   shipper_id                566067 non-null  object 
 1   order_id                  566067 non-null  int64  
 2   ship_date                 566067 non-null  object 
 3   delivery_date             566067 non-null  object 
 4   shipping_fee              566067 non-null  float64
 5   shipper_company           566067 non-null  object 
 6   shipper_vehicle           566067 non-null  object 
 7   shipper_experience_years  566067 non-null  int64  
 8   shipper_rating            566067 non-null  float64
 9   delivery_success_rate     566067 non-null  float64
 10  average_delivery_time     566067 non-null  int64  
 11  working_shift             566067 non-null  object 
 12  join_date                 566067 n

,order_id,ship_date,delivery_date,shipping_fee,shipper_id
0,1,2012-07-07,2012-07-11,1.37,SHP00001
1,2,2012-07-06,2012-07-10,2.60,SHP00002
2,3,2012-07-04,2012-07-07,2.38,SHP00003
3,4,2012-07-05,2012-07-11,2.49,SHP00004
4,6,2012-07-09,2012-07-16,25.79,SHP00005



Dữ liệu SHIPMENT đã vượt qua các kiểm tra cơ bản!

Đã lưu file:
/content/Shipment_clean.csv


,order_id,ship_date,delivery_date,shipping_fee,shipper_id
0,1,2012-07-07,2012-07-11,1.37,SHP00001
1,2,2012-07-06,2012-07-10,2.60,SHP00002
2,3,2012-07-04,2012-07-07,2.38,SHP00003
3,4,2012-07-05,2012-07-11,2.49,SHP00004
4,6,2012-07-09,2012-07-16,25.79,SHP00005
...,...,...,...,...,...
566062,834196,2022-12-29,2022-12-31,2.22,SHP00078
566063,834293,2022-12-29,2022-12-31,26.83,SHP00003
566064,834299,2022-12-28,2022-12-31,1.23,SHP00012
566065,834314,2022-12-29,2022-12-31,2.28,SHP00039
